# DES Hub Netbacks - Historical Moneyness Forward Curve

Calling the Spark v1.0 DES Hub Netbacks endpoint and persisting SparkNWE / SparkSWE forward curves to produce a daily DES LNG forward curve per terminal. Moneyness = DES Hub Netback (Variable Regas Costs Only) minus the relevant regional marker.

For a full explanation of how to import our DES Hub Netbacks or SparkNWE data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html


### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint: https://www.sparkcommodities.com/api/lng-access/des-hub-netbacks.html

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from pprint import pprint
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):
    """
    Get a new access_token. Access tokens are the thing that applications use to make
    API requests. Access tokens must be kept confidential in storage.

    # Procedure:

    Do a POST query with `grantType` and `scopes` in the body. A basic authorization
    HTTP header is required. The "Basic" HTTP authentication scheme is defined in
    RFC 7617, which transmits credentials as `clientId:clientSecret` pairs, encoded
    using base64.
    """

    # Note: for the sake of this example, we choose to use the Python urllib from the
    # standard lib. One should consider using https://requests.readthedocs.io/

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
        "scopes": "read:access,read:prices"
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not downloaded them already. Instructions for downloading your credentials can be found here:

https://app.sparkcommodities.com/data-integrations/api

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

## 2. Reference Data

The reference-data endpoint lists every DES Hub Netbacks terminal, along with its `TerminalUUID` and the Gas Hubs used as the front-month and forward-curve price sources. Use the UUID whenever you want to filter subsequent calls to a single terminal.

In [ ]:
## Defining the reference-data calling function

def fetch_deshub_reference_data(access_token, format='json'):

    uri = "/v1.0/lng/access/des-hub-netbacks/reference-data/"
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token, format=format)

    if format == 'json':
        data = content
    elif format == 'csv':
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))

    return data

In [ ]:
# fetching in CSV format (easier to browse the terminal list)
ref_data = fetch_deshub_reference_data(access_token, format='csv')
ref_data

## 3. Historical Data

The historical endpoint returns DES Hub Netbacks releases for any date range (max 365 days per call).

In [ ]:
## Defining the function

def fetch_historical_price_releases(access_token, unit, start=None, end=None,
                                    terminal_uuid=None, limit=None, offset=None,
                                    format='json'):

    query_params = "?unit={}".format(unit)

    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)
    if terminal_uuid is not None:
        query_params += "&terminal-uuid={}".format(terminal_uuid)
    if limit is not None:
        query_params += "&limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)

    uri = "/v1.0/lng/access/des-hub-netbacks/{}".format(query_params)
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token, format=format)

    if format == 'json':
        data = content
    elif format == 'csv':
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))

            for c in list(data.columns)[8:]:
                data[c] = pd.to_numeric(data[c], errors='coerce')
            data['ReleaseDate'] = pd.to_datetime(data['ReleaseDate'])
            data['DeliveryMonth'] = pd.to_datetime(data['DeliveryMonth'])

    return data

## N.B. Historical Data Limits

A single call to the historical endpoint is limited to 365 days of data. If you need a longer range, use the loop function below, which calls the endpoint one year at a time and concatenates the results into a single Pandas DataFrame.

In [ ]:
def loop_historical_data(access_token, hist_unit, hist_start, hist_end):

    # input validation
    start_dt = datetime.datetime.strptime(hist_start, '%Y-%m-%d')
    end_dt   = datetime.datetime.strptime(hist_end,   '%Y-%m-%d')

    if start_dt >= end_dt:
        raise ValueError(
            "hist_start ({}) must be strictly before hist_end ({})".format(hist_start, hist_end)
        )

    hist_diff = (end_dt - start_dt).days
    t = 0
    starts = []
    ends = []

    # window size - 1 year per call
    w = 365

    while t < hist_diff:
        # initialising dataframe
        if t == 0:
            # bound the first window by hist_end so we never request beyond it
            diff_end = min(start_dt + pd.Timedelta(days=w), end_dt).strftime('%Y-%m-%d')
            hist_df = fetch_historical_price_releases(
                access_token, unit=hist_unit, start=hist_start, end=diff_end, format='csv'
            )
            starts.append(hist_start)
            ends.append(diff_end)

        # appending additional historical data
        else:
            diff_start_dt = start_dt + pd.Timedelta(days=t + 1)
            diff_start = diff_start_dt.strftime('%Y-%m-%d')
            # bound every subsequent window by hist_end
            diff_end = min(diff_start_dt + pd.Timedelta(days=w), end_dt).strftime('%Y-%m-%d')

            historical_addition = fetch_historical_price_releases(
                access_token, unit=hist_unit, start=diff_start, end=diff_end, format='csv'
            )
            try:
                hist_df = pd.concat([hist_df, historical_addition])
            except Exception as e:
                # fallback if the first slice came back empty
                print(e)
                hist_df = historical_addition.copy()
            starts.append(diff_start)
            ends.append(diff_end)

        t += w

    hist_df = hist_df.drop_duplicates().reset_index(drop=True)

    return hist_df

In [ ]:
# Input start and end dates
today = datetime.datetime.now().strftime('%Y-%m-%d')
#hist_start = (datetime.datetime.now() - datetime.timedelta(days=365)).strftime('%Y-%m-%d')
hist_start = '2026-02-01'

# Call Data
hist_df = loop_historical_data(access_token, hist_unit='usd-per-mmbtu',
                                hist_start=hist_start, hist_end=today)

hist_df.head(5)

## 4. Sinking Regas Costs Components

Output columns added to `hist_df`:
- `NetbackTTFBasisSunk`
- `NetbackOutrightSunk`


In [ ]:
# Global sunk slot components - added back for every terminal
sunk_slot_cols = ['SlotLtCapacityEstimate', 'SlotUnloadStorageRegas', 'SlotBerth', 'SlotBerthUnloadStorageRegas']

# Per-terminal extra sunk costs - layered on top of the global sunk costs defined above
# Value is a list of cost columns to treat as sunk for that terminal only
# Edit this dict to suit your analysis. Unlisted terminals use the global set.

sunk_terminal_costs = {
    # 'Isle of Grain': ['EntryCapacity', 'EntryVariable'],
    # 'South Hook': ['EntryCapacity'],
}

# Sink global sunk costs across all terminals
sunk_add_back = hist_df[sunk_slot_cols].fillna(0).sum(axis=1)

# Sink terminal-specific sunk costs
for terminal, cols in sunk_terminal_costs.items():
    if not cols:
        continue
    missing = [c for c in cols if c not in hist_df.columns]
    if missing:
        raise KeyError('sunk_terminal_costs[{!r}] references missing columns: {}'.format(terminal, missing))
    mask = hist_df['TerminalName'] == terminal
    if mask.any():
        sunk_add_back.loc[mask] += hist_df.loc[mask, cols].fillna(0).sum(axis=1)

hist_df['NetbackTTFBasisSunk'] = hist_df['NetbackTTFBasis'] + sunk_add_back
hist_df['NetbackOutrightSunk'] = hist_df['NetbackOutright'] + sunk_add_back

hist_df[['ReleaseDate', 'TerminalName', 'DeliveryMonthIndex', 'NetbackTTFBasis',
         'NetbackTTFBasisSunk']].head(10)


### Pivot into a wide DataFrame - one column per terminal

For the moneyness merge downstream we want a wide-format table: one row per (ReleaseDate, DeliveryMonth) with a column per terminal containing its `NetbackTTFBasisSunk`.

In [ ]:
# one row per (ReleaseDate, DeliveryMonth), one column per terminal
terminal_list = list(hist_df['TerminalName'].unique())

id_cols = ['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', 'DeliveryMonthName']
value_col = 'NetbackTTFBasisSunk'

deshub_final = (
    hist_df
    .pivot_table(index=id_cols, columns='TerminalName', values=value_col, aggfunc='first')
    .reset_index()
)
deshub_final.columns.name = None

# restore terminal-appearance ordering (pivot_table sorts alphabetically by default)
deshub_final = deshub_final[id_cols + terminal_list]
deshub_final.head(5)

## 5. Classification Dictionaries

Maps each terminal (keyed by its v1.0 `TerminalName`) to either `nwe` or `swe`, which in turn controls which Spark regional marker gets subtracted in the moneyness calculation.

A second dictionary maps terminals to countries for optional country-level aggregation.

In [ ]:
# Region mapping 
terminal_region_dict = {
    'Gate': 'nwe',
    'Isle of Grain': 'nwe',
    'Zeebrugge': 'nwe',
    'South Hook': 'nwe',
    'Dunkerque': 'nwe',
    'Le Havre': 'nwe',
    'Montoir': 'nwe',
    'EemsEnergyTerminal': 'nwe',
    'Brunsbuttel': 'nwe',
    'Deutsche Ostsee': 'nwe',
    'Deutsche Ostsee Phase 1': 'nwe',
    'Wilhelmshaven 1': 'nwe',
    'Wilhelmshaven 2': 'nwe',
    'Stade': 'nwe',
    'Fos Cavaou': 'swe',
    'Adriatic': 'swe',
    'OLT Toscana': 'swe',
    'Piombino': 'swe',
    'Ravenna': 'swe',
    'Spain TVB': 'swe',
}

# Country mapping
terminal_country_dict = {
    'Netherlands': ['Gate', 'EemsEnergyTerminal'],
    'UK':          ['Isle of Grain', 'South Hook'],
    'Belgium':     ['Zeebrugge'],
    'France':      ['Dunkerque', 'Le Havre', 'Montoir', 'Fos Cavaou'],
    'Germany':     ['Brunsbuttel', 'Deutsche Ostsee',
                    'Wilhelmshaven 1', 'Wilhelmshaven 2', 'Stade'],
    'Italy':       ['Adriatic', 'OLT Toscana', 'Piombino', 'Ravenna'],
    'Spain':       ['Spain TVB'],
}

## 6. SparkNWE & SparkSWE - Data Import

Calling the SparkNWE and SparkSWE front-month + forward-curve cargo contracts (TTF basis) to compare against the terminal DES Hub netbacks. Moneyness = DES Hub Netback (Var Regas Only) - regional marker.

In [ ]:
# Defining function to call contracts endpoint to import cargo data
def fetch_cargo_releases(access_token, ticker, limit=4, offset=None):

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)

    uri = "/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params)
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token)

    return content['data']


# call front month and forward curve, combine into one DataFrame per release date
def cargo_focurve(access_token, ticker, limit):

    f_tick = ticker + '-b-f'
    fo_tick = ticker + '-b-fo'

    f_data = fetch_cargo_releases(access_token, f_tick, limit)
    fo_data = fetch_cargo_releases(access_token, fo_tick, limit)

    release_dates = []
    period_start = []
    contract_ids = []
    spark = []

    # single-release convenience path (latest only)
    if limit == 1:
        release_date = f_data[0]["releaseDate"]
        contract_ids.append(f_data[0]['contractId'])
        release_dates.append(release_date)

        period_start_at = f_data[0]['data'][0]['dataPoints'][0]["deliveryPeriod"]["startAt"]
        period_start.append(period_start_at)
        spark.append(f_data[0]['data'][0]['dataPoints'][0]['derivedPrices']['usdPerMMBtu']['spark'])

        for release in fo_data:
            for data_point in release['data'][0]['dataPoints']:
                contract_ids.append(release['contractId'])
                release_dates.append(release["releaseDate"])
                period_start.append(data_point["deliveryPeriod"]["startAt"])
                spark.append(data_point['derivedPrices']['usdPerMMBtu']['spark'])

    # full historical path: align each front month release with the most recent
    # forward curve release on or before it
    else:
        cl = 0

        for l in range(limit):
            try:
                release_date = f_data[l]["releaseDate"]
            except (IndexError, KeyError):
                break

            contract_ids.append(f_data[l]['contractId'])
            release_dates.append(release_date)
            release_datetime = datetime.datetime.strptime(release_date, '%Y-%m-%d')

            front_period_start_at = f_data[l]['data'][0]['dataPoints'][0]["deliveryPeriod"]["startAt"]
            period_start.append(front_period_start_at)
            spark.append(f_data[l]['data'][0]['dataPoints'][0]['derivedPrices']['usdPerMMBtu']['spark'])

            # guard against running off the end of fo_data
            if cl >= len(fo_data):
                continue

            fo_datetime = datetime.datetime.strptime(fo_data[cl]['releaseDate'], '%Y-%m-%d')

            # assessment date - front month AND forward curve both published today
            if release_datetime == fo_datetime:
                for data_point in fo_data[cl]['data'][0]['dataPoints']:
                    contract_ids.append(fo_data[cl]['contractId'])
                    release_dates.append(release_date)
                    period_start.append(data_point["deliveryPeriod"]["startAt"])
                    spark.append(data_point['derivedPrices']['usdPerMMBtu']['spark'])
                cl += 1

            # non-assessment date - persist most recent published forward curve
            # but drop the data point that overlaps with today's front month
            else:
                for data_point in fo_data[cl]['data'][0]['dataPoints']:
                    period_start_at = data_point["deliveryPeriod"]["startAt"]
                    if period_start_at != front_period_start_at:
                        contract_ids.append(fo_data[cl]['contractId'])
                        release_dates.append(release_date)
                        period_start.append(period_start_at)
                        spark.append(data_point['derivedPrices']['usdPerMMBtu']['spark'])

    hist_cargo_df = pd.DataFrame({
        'Release Date': release_dates,
        'ticker': contract_ids,
        'Period Start': period_start,
        'Price': spark,
    })

    hist_cargo_df['Price'] = pd.to_numeric(hist_cargo_df['Price'])
    hist_cargo_df['Release Date'] = pd.to_datetime(hist_cargo_df['Release Date']).dt.tz_localize(None)

    return hist_cargo_df

In [ ]:
# calling the data
nwe_fo_df = cargo_focurve(access_token, 'sparknwe', limit=5000)
swe_fo_df = cargo_focurve(access_token, 'sparkswe', limit=5000)

In [ ]:
# Merging the NWE and SWE datasets into a single dataframe
cargo_df = pd.merge(nwe_fo_df, swe_fo_df, on=['Release Date', 'Period Start'], how='left')
cargo_df = cargo_df.rename(columns={
    'Price_x': 'SparkNWE',
    'Price_y': 'SparkSWE',
})
cargo_df = cargo_df.drop(columns=['ticker_x', 'ticker_y'])

# cut off data prior to daily front-month assessments for NWE (switchover 2023-11-23)
switch_date = '2023-11-23' # Do not edit (value should be 2023-11-23)
date_limit = cargo_df[cargo_df['Release Date'] == switch_date].index[-1]
cargo_df = cargo_df.iloc[:date_limit + 1].copy()

# Persist SparkSWE forward curves across non-assessment days (SWE fo is weekly, NWE is daily) 
# Groupby 'Period Start' to make sure data fills match against Period Start
# limit=14 covers potential bank holiday gaps.
cargo_df['Period Start'] = pd.to_datetime(cargo_df['Period Start'])
cargo_df = cargo_df.sort_values(['Period Start', 'Release Date']).reset_index(drop=True)
cargo_df['SparkSWE'] = cargo_df.groupby('Period Start')['SparkSWE'].ffill(limit=14)

# Order by release date
cargo_df = cargo_df.sort_values(['Release Date', 'Period Start'], ascending=[False, True]).reset_index(drop=True)

## 7. Final Moneyness Table

Moneyness = DES Hub Netback (w/ or w/o Sunk Costs) - Spark regional marker, where the marker is SparkNWE (NWE terminals) or SparkSWE (SWE terminals).

In [ ]:
# Vectorised moneyness calculation.

# only consider terminals mapped to a region
new_terminal_list = [t for t in terminal_list if t in terminal_region_dict]
unmapped = [t for t in terminal_list if t not in terminal_region_dict]
if unmapped:
    print('Skipping terminals not mapped to a region in terminal_region_dict: {}'.format(unmapped))

# merge DES Hub netbacks with NWE/SWE data 
money_merge = pd.merge(
    deshub_final,
    cargo_df[['Release Date', 'Period Start', 'SparkNWE', 'SparkSWE']],
    left_on=['ReleaseDate', 'DeliveryMonth'],
    right_on=['Release Date', 'Period Start'],
    how='left',
)

# initialise output
money_df = money_merge[['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', 'DeliveryMonthName']].copy()

# compute moneyness per terminal
nwe_terms = [t for t in new_terminal_list if terminal_region_dict[t] == 'nwe']
swe_terms = [t for t in new_terminal_list if terminal_region_dict[t] == 'swe']

# Subtract either NWE or SWE marker depending on terminal
for t in nwe_terms:
    money_df[t] = money_merge[t] - money_merge['SparkNWE']
for t in swe_terms:
    money_df[t] = money_merge[t] - money_merge['SparkSWE']

# round to 3 d.p.
money_df[new_terminal_list] = money_df[new_terminal_list].round(3)

In [ ]:
# Filtering data to view latest forward curve
latest = money_df['ReleaseDate'].unique()[-1]
money_df[money_df['ReleaseDate'] == latest]

# Average by country

In [ ]:
# Function to average by country

id_cols = ['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', 'DeliveryMonthName']

def aggregate_by_country(df, country_dict, id_cols=id_cols, decimals=3):
    source = df                       # keep original (has terminal columns)
    out = df[id_cols].copy()          # output starts with just id cols

    # Iterate through countries
    for country, terminals in country_dict.items():
        # Only use terminals actually present as columns
        present = [t for t in terminals if t in source.columns]   # check ORIGINAL
        if not present:
            # No terminals available for this country in this df — emit NaN column
            out[country] = np.nan
            continue
        # Average over all relevant terminals for country
        out[country] = source[present].mean(axis=1, skipna=True).round(decimals)
    return out


# Country-level moneyness (averaged from money_df)
country_money = aggregate_by_country(money_df, terminal_country_dict)

# Country-level netback (averaged from deshub_final, i.e. NetbackTTFBasisSunk)
country_df = aggregate_by_country(deshub_final, terminal_country_dict)

# 8. Plotting

## 8.1. Plotting Front Month Moneyness Evolution for several terminals

In [ ]:
# Front month and forward curve dataframes for terminals
frontmonth_term = money_df[money_df['DeliveryMonthIndex'] == 'M+1'].copy()
latest_term = money_df[money_df['ReleaseDate'] == latest].copy()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

fig, ax= plt.subplots(figsize=(14,7))

ax.hlines(0, frontmonth_term['ReleaseDate'].iloc[0], latest_term['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20), color='grey', zorder=0)

# User input for which countries to plot, and associated plot colours
my_terminals = ['Brunsbuttel', 'EemsEnergyTerminal', 'Zeebrugge', 'Montoir', 'Spain TVB', 'Gate']
colours = ['black', '#0086ad', 'forestgreen', '#00c2c7', 'firebrick', 'orange']

print("Latest Assessment: " + str(frontmonth_term['ReleaseDate'].iloc[0]))

# Iterating through all chosen terminals
cz = 0
for t in my_terminals:
    # Print latest Moneyness value
    print(t + ' = ' + str(frontmonth_term[t].iloc[-1]))

    # Plot front month and forward curve
    ax.scatter(frontmonth_term['ReleaseDate'].iloc[-1],  frontmonth_term[t].iloc[-1], color=colours[cz], marker='o', s=80)
    ax.plot(frontmonth_term['ReleaseDate'], frontmonth_term[t], linewidth=2.0, label=t, color=colours[cz])
    ax.plot(latest_term['DeliveryMonth'].iloc[1:], latest_term[t].iloc[1:], linewidth=2.0, marker='s', linestyle='--', color=colours[cz])
    
    cz+=1

# Conditional <0 red shading
negrange = [frontmonth_term['ReleaseDate'].iloc[0]-pd.Timedelta(days=1), latest_term['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20)]
ax.fill_between(negrange, 0, -30, color='red', alpha=0.05)

# Axis params
plt.xlim([frontmonth_term['ReleaseDate'].iloc[0]-pd.Timedelta(days=1), latest_term['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20)])
plt.ylim(money_df[my_terminals].min().min(), money_df[my_terminals].max().max())
plt.tight_layout()
ax.legend(loc='upper right', frameon=True, edgecolor='#dddddd', fontsize=9.5)

plt.show()

## 8.2. Plotting Country-Averaged Moneyness

In [ ]:
# Front month and forward curve dataframes for countries
frontmonth_country = country_money[country_money['DeliveryMonthIndex'] == 'M+1'].copy()
latest_country = country_money[country_money['ReleaseDate'] == latest].copy()

In [ ]:
fig, ax= plt.subplots(figsize=(14,7))

ax.hlines(0, frontmonth_country['ReleaseDate'].iloc[0], latest_country['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20), color='grey', zorder=0)

# User input for which countries to plot, and associated plot colours
my_countries = ['UK', 'France', 'Italy', 'Germany', 'Spain', 'Netherlands']
colours = ['#0086ad', '#00c2c7', 'forestgreen', 'black', 'firebrick', 'orange']

print("Latest Assessment: " + str(frontmonth_country['ReleaseDate'].iloc[0]))

# Iterating through all chosen countries
cz = 0
for t in my_countries:
    # Print latest Moneyness value
    print(t + ' = ' + str(frontmonth_country[t].iloc[-1]))

    # Plot front month and forward curve
    ax.scatter(frontmonth_country['ReleaseDate'].iloc[-1],  frontmonth_country[t].iloc[-1], color=colours[cz], marker='o', s=80)
    ax.plot(frontmonth_country['ReleaseDate'], frontmonth_country[t], linewidth=2.0, label=t, color=colours[cz])
    ax.plot(latest_country['DeliveryMonth'].iloc[1:], latest_country[t].iloc[1:], linewidth=2.0, marker='s', linestyle='--', color=colours[cz])
    
    cz+=1

# Conditional <0 red shading
negrange = [frontmonth_country['ReleaseDate'].iloc[0]-pd.Timedelta(days=1), latest_country['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20)]
ax.fill_between(negrange, 0, -30, color='red', alpha=0.05)

# Axis params
plt.xlim([frontmonth_country['ReleaseDate'].iloc[0]-pd.Timedelta(days=1), latest_country['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20)])
plt.ylim(country_money[my_countries].min().min(), country_money[my_countries].max().max())
plt.tight_layout()
ax.legend(loc='lower right', frameon=True, edgecolor='#dddddd', fontsize=11.5)

plt.show()

## 8.3. Plotting Custom Month Historical Evolution

In [ ]:
# Choose Delivery Month of interest
month = 'Sep26'

month_term = money_df[money_df['DeliveryMonthName'] == month].copy()
month_term.head(3)

In [ ]:
fig, ax= plt.subplots(figsize=(14,7))

ax.hlines(0, month_term['ReleaseDate'].iloc[0], month_term['ReleaseDate'].iloc[-1]+pd.Timedelta(days=20), color='grey', zorder=0)

# User input for which countries to plot, and associated plot colours
my_terminals = ['Brunsbuttel', 'EemsEnergyTerminal', 'Zeebrugge', 'Montoir', 'Spain TVB', 'Gate']
colours = ['black', '#0086ad', 'forestgreen', '#00c2c7', 'firebrick', 'orange']

print("Latest Assessment: " + str(month_term['ReleaseDate'].iloc[0]))

# Iterating through all chosen terminals
cz = 0
for t in my_terminals:
    # Print latest Moneyness value
    print(t + ' = ' + str(month_term[t].iloc[-1]))

    # Plot historical evolution
    ax.scatter(month_term['ReleaseDate'].iloc[-1],  month_term[t].iloc[-1], color=colours[cz], marker='o', s=80)
    ax.plot(month_term['ReleaseDate'], month_term[t], linewidth=2.0, label=t, color=colours[cz])
    
    cz+=1

# Conditional <0 red shading
negrange = [month_term['ReleaseDate'].iloc[0] -pd.Timedelta(days=5), month_term['ReleaseDate'].iloc[-1]+pd.Timedelta(days=5)]
ax.fill_between(negrange, 0, -30, color='red', alpha=0.05)

# Axis params
plt.xlim([month_term['ReleaseDate'].iloc[0]-pd.Timedelta(days=1), month_term['ReleaseDate'].iloc[-1]+pd.Timedelta(days=5)])
plt.ylim(month_term[my_terminals].min().min(), month_term[my_terminals].max().max())
plt.tight_layout()
ax.legend(loc='upper right', frameon=True, edgecolor='#dddddd', fontsize=9.5)
plt.title('European Terminal Moneyness - ' + month)

plt.show()

## 8.4. Plotting Forward Curve Evolution for a specific terminal

In [ ]:
my_terminal = 'Montoir'
term_df = money_df[['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', my_terminal]].copy().dropna()

available_rels = list(term_df['ReleaseDate'].unique())
print('Earliest Date: ' + str(available_rels[0]))
print('Latest Date: ' + str(available_rels[-1]))

In [ ]:
# Manually define which release dates to compare forward curves for
#plot_dates = ['2025-01-02', '2025-03-03', '2025-05-02', '2025-07-01', latest]

# Alternative: auto-pick N release dates evenly spaced between the earliest and latest available forward curves
n_curves = 4
plot_dates = [str(d)[:10] for d in 
            np.sort(term_df['ReleaseDate'].unique())[np.linspace(0, term_df['ReleaseDate'].nunique()-1, n_curves).round().astype(int)]]

# Checking each release date is valid & has an associated forward curve
fo_curves = []
for d in plot_dates:
    curve = term_df[term_df['ReleaseDate'] == d][['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', my_terminal]]
    if len(curve) == 0:
        print('No data for release date {} - skipping'.format(d))
        continue
    fo_curves.append(curve) 
if not fo_curves:
    raise ValueError('No release dates from plot_dates found in term_df')

# setting ranges of alphas, linewidths and markersizes for each curve
alphas = np.linspace(0.8, 1, len(fo_curves))
linewidths = np.linspace(0.7, 2, len(fo_curves))
msizes = np.linspace(6, 12, len(fo_curves))

# Setting up axis
fig, ax= plt.subplots(figsize=(16,7))

# anchor x-axis to the latest plotted curve's earliest delivery month
x_start = fo_curves[-1]['DeliveryMonth'].iloc[0]

# Plotting forward curves, but only for contract months related to latest curve
for f, fo in enumerate(fo_curves):
    release_label = str(fo['ReleaseDate'].iloc[0])[:10]   # get release date to label curve
    fo = fo[fo['DeliveryMonth'] >= x_start]               # only retrieve months that match those seen in the latest curve
    
    if fo.empty:                                           # in case curve ends before x_start
        print('No overlap with x_start for release {} - skipping'.format(release_label))
        continue
    
    ax.plot(fo['DeliveryMonth'], fo[my_terminal], linestyle='--', marker='s',
        label=release_label, linewidth=linewidths[f], alpha=alphas[f], markersize=msizes[f])

# Plot y=0 line to make the In the Money/Out the Money boundary clear
ax.hlines(0, x_start-pd.Timedelta(days=40), term_df['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20), color='grey')

# Shading <0 area of the chart
negrange = [x_start -pd.Timedelta(days=20), term_df['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20)]
ax.fill_between(negrange, 0, -30.0, color='red', alpha=0.05)

plt.title('Moneyness Forward Curve Historical Evolution - ' + my_terminal)
plt.xlim([x_start-pd.Timedelta(days=20), term_df['DeliveryMonth'].iloc[-1]+pd.Timedelta(days=20)])
plt.ylim(min(term_df[my_terminal]), max(term_df[my_terminal]))
plt.tight_layout()
ax.legend(loc='lower right', frameon=True, edgecolor='#dddddd', fontsize=11.5)
plt.show()